<a href="https://colab.research.google.com/github/iam4tart/speech-lab/blob/main/03-true-streaming-causal-model/causal_types.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn.functional as F

d_k is vector length (embedding dimension) of Q and K


d_k = Q.shape[-1] or K.shape[-1] by construction

In [ ]:
# usual bidirectional self-attention - sees everything
def attention(x, Wq, Wk, Wv):
  Q, K, V = x @ Wq, x @ Wk, x @ Wv
  d_k = Q.shape[-1]
  scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)
  weights = F.softmax(scores, dim=-1)
  return weights @ V

In [ ]:
# causal attention - blocks the future
def causalAttention(x, Wq, Wk, Wv):
  Q, K, V = x @ Wq, x @ Wk, x @ Wv
  d_k = Q.shape[-1]
  scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)

  # x = (T, D)
  T = x.shape[0]
  future_mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
  scores = scores.masked_fill(future_mask, float('-inf'))

  weights = F.softmax(scores, dim=-1)
  return weights @ V

In [ ]:
# sliding window attention - blocks the future and the distant past
def sliding_window_attention(x, Wq, Wk, Wv, window=3):
  Q, K, V = x @ Wq, x @ Wk, x @ Wv
  d_k = Q.shape[-1]
  scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)

  T = x.shape[0]
  future_mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
  distant_past = torch.tril(torch.ones(T, T), diagonal=-window).bool()
  scores = scores.masked_fill(future_mask | distant_past, float('-inf'))

  weights = F.softmax(scores, dim=-1)
  return weights @ V